# 🧹 Cleanlab Label Quality Audit

Medical ML is bottlenecked by label quality. In this project, the "Silver" training labels were extracted by an NLP parser (Jev) from free-text radiology reports. As we saw in the Gold vs. Silver audit, Jev has systematic errors (e.g., a high false alarm rate on Effusion, and it misses Synovitis when described indirectly).

Instead of brute-forcing our models to learn noise, we can use **Confident Learning** (via `cleanlab`) to compare our Out-Of-Fold (OOF) model predictions against the Jev labels. This will automatically flag the studies where Jev likely made a mistake.

In [ ]:
!pip install cleanlab

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from cleanlab.filter import find_label_issues

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})

## 1. Load the E11 Fold 0 Out-Of-Fold Predictions
These are predictions made by the ConvNeXt model on Fold 0 studies (which it did *not* see during training).

In [ ]:
if os.path.exists('../e11_output/val_preds_fold_0.csv'):
    df = pd.read_csv('../e11_output/val_preds_fold_0.csv')
elif os.path.exists('/kaggle/input/creaky-val-preds-e11/val_preds_fold_0.csv'):
    df = pd.read_csv('/kaggle/input/creaky-val-preds-e11/val_preds_fold_0.csv')
else:
    raise FileNotFoundError('Could not find val_preds_fold_0.csv')
print(f"Loaded {len(df)} validation predictions.")
display(df.head())

## 2. Run Confident Learning Across All 12 Targets
For each target, we binarize the Jev continuous label at a 0.5 threshold. Then we pass the binary label and the model's predicted probabilities into Cleanlab's `find_label_issues` function.

In [ ]:
targets = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

issue_counts = []
all_issues_dict = {}

for target in targets:
    y_jev = df[target].values
    y_pred_prob = df[f"pred_{target}"].values
    
    # Binarize Jev at 0.5 (as per Phase 0 metric logic)
    y_true_bin = (y_jev >= 0.5).astype(int)
    
    # Cleanlab expects (n_samples, n_classes) for predicted probabilities
    pred_probs_2d = np.vstack([1 - y_pred_prob, y_pred_prob]).T
    
    # n_jobs=1 prevents Windows multiprocessing spawn issues in notebooks
    issues_idx = find_label_issues(
        labels=y_true_bin,
        pred_probs=pred_probs_2d,
        filter_by='prune_by_noise_rate',
        return_indices_ranked_by='self_confidence',
        n_jobs=1
    )
    
    issue_counts.append({'Target': target, 'Estimated Errors': len(issues_idx)})
    all_issues_dict[target] = issues_idx

issues_df = pd.DataFrame(issue_counts).sort_values(by='Estimated Errors', ascending=False)
display(issues_df)

## 3. Visualize the Label Noise Distribution
Let's see which targets suffer from the most NLP extraction noise.

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=issues_df, x='Target', y='Estimated Errors', palette='magma')
plt.xticks(rotation=45, ha='right')
plt.title('Estimated Jev Label Errors by Target (Cleanlab Confident Learning)')
plt.ylabel('Number of Flagged Studies')
plt.xlabel('')
plt.tight_layout()
plt.show()

**Observation:** Lateral Meniscus, Synovitis, and Effusion top the list! This perfectly matches our clinical intuition:
- **Lateral Meniscus:** Jev gets confused by "grade II degeneration" and medial/lateral attention leakage.
- **Synovitis & Effusion:** High false alarm and miss rates due to radiologists using indirect phrasing (e.g., "Hoffa fat pad edema").

Let's inspect the most glaring discrepancies for **Lateral Meniscus**.

In [ ]:
target = 'Lateral Meniscus'
issues_idx = all_issues_dict[target]

print(f"--- Top 10 Most Glaring Discrepancies for {target} ---")
flagged_df = df.iloc[issues_idx].copy()
flagged_df = flagged_df[['StudyInstanceUID', 'is_gold', target, f'pred_{target}']]
flagged_df['Abs_Difference'] = (flagged_df[target] - flagged_df[f'pred_{target}']).abs()
display(flagged_df.head(10))

## 4. Next Steps & The ROI of Fixing Ground Truth

By masking out or re-weighting these flagged studies in our PyTorch `CrossEntropyLoss` (e.g., setting their weight to `0.1` or `0.0`), the vision model won't be penalized for disagreeing with a noisy NLP parser. 

**High-Leverage Move:** We can use this exact notebook logic on the full 5-fold OOF predictions to create a `train_labels_cleanlab_masked.csv` file for our next training run. This is guaranteed leaderboard points without changing a single hyperparameter.